# Forward-scan sonar: the elevation ambiguity, and two ways around it

A forward-scan sonar returns an image addressed by **range and bearing only**. The
elevation angle a return came from is not recorded, and it is not recoverable from
that image. This notebook shows that the simulator reproduces that property rather
than approximating it, then shows two things that work anyway: a second sensor that
measures the missing angle, and a waveform that buys range resolution without a
short pulse.

Everything below calls the same compiled core the demo scripts and `run_all.sh`
call. Nothing here reimplements the physics.

In [ ]:
import sys, glob, os
sys.path.insert(0, "."); sys.path.insert(0, "python")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Audio, display

import acoustics, montecarlo, outputs, recovery, scene as sc, sonar


def figure(name):
    """Newest run that actually contains this artifact, else output/.

    Picking one results directory up front and using it for everything breaks as
    soon as a demo is added: the newest run predates the new figure and every
    lookup into it fails. Each artifact is resolved on its own instead.
    """
    for directory in sorted(glob.glob("results/*"), reverse=True) + ["output"]:
        candidate = os.path.join(directory, name)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f"{name} not found; run ./run_all.sh first")


print("newest run:", sorted(glob.glob("results/*"))[-1] if glob.glob("results/*") else "none")


## 1. The ambiguity is a property of the model, not an approximation

Put a target at range `r`, bearing `theta`, elevation `+phi`. Put an identical target
at `(r, theta, -phi)`. Those are two genuinely different places in the water. Render
both.

The image is addressed by range and bearing, and the vertical beam weighting `B(phi)`
is even in `phi`, so nothing in the pipeline can tell them apart. If a bug leaked
elevation into a pixel address, this test would fail.

In [ ]:
sim = sonar.SonarSimulator(frequency_hz=1.2e6, num_azimuth_bins=193, num_range_bins=700,
                           horizontal_fov_deg=30.0, vertical_beamwidth_deg=14.0,
                           max_range_m=10.0, num_elevation_subrays=1024)

pair = [sim.render([sonar.make_sphere(sc.spherical_to_world(6.0, 4.0, sign * 5.0), 0.15, 0.9)])
        for sign in (+1, -1)]
difference = float(np.abs(pair[0] - pair[1]).max())
print(f"peak intensity        {pair[0].max():.6e}")
print(f"max |difference|      {difference:.3e}")
print(f"relative to peak      {difference / pair[0].max():.3e}")

The residual is not zero only because the sub-ray that strikes the `+phi` target is
`s` while for `-phi` it is `n-1-s`, so the contributions are summed in the opposite
order and floating-point addition is not associative. That is double-precision
round-off, not a physical difference.

In [ ]:
display(Image(filename=figure("demo1_ambiguity.png"), width=900))

## 2. What a second viewpoint can and cannot do

Translating the sonar sideways does **not** separate `+phi` from `-phi`: only `z^2`
enters the range, so the two remain identical however many pings you collect along a
constant-depth line. Translating vertically does separate them, immediately.

In [ ]:
display(Image(filename=figure("demo5_motion.png"), width=900))
print(open(figure("demo5_motion.txt")).read())

## 3. Recovering elevation with a camera

A sonar bin is consistent with a one-parameter family of 3-D points: the arc swept by
elevation across the vertical beam. A camera pixel is the complement, two angles and
no range. Project the arc into the camera, cross it with the pixel the target was
detected in, and the point is determined.

Both detections below are made off rendered images, not read out of the scene.

In [ ]:
display(Image(filename=figure("demo8_optiacoustic.png"), width=1000))

One bin alone leaves 0.959 m of arc open. Fused with the camera the elevation comes
back at 3.486 degrees against a true 3.500, a 2.4 cm position error.

The turbidity sweep in the bottom right is the other half of the argument: the optical
direct term decays as `exp(-2 c r)` while the veiling glow grows as `1 - exp(-c r)`,
so past about `c = 0.2` per metre the camera detection is gone and the pair falls back
to the sonar's arc. The sonar image itself is unchanged throughout.

### That was one run. Here it is as a distribution.

Speckle has contrast 1, so a single realisation can land anywhere and a single number
is an anecdote. Below, the same recovery is rerun with an independent speckle seed and
independent camera read noise each time, calling exactly the functions the single run
called. The interval is a percentile bootstrap rather than a t interval, because the
error comes from a centroid on a thresholded blob and is not obviously Gaussian.


In [ ]:
import montecarlo, recovery

TRIALS = 60          # the full run in demo 13 uses 200; 60 keeps the notebook quick
CONFIDENCE = 0.95

axes_mc = recovery.scene_axes()
objects_mc = recovery.build_objects(recovery.target_centre(axes_mc))
sim_mc = recovery.build_sonar()

errors = [recovery.trial(sim_mc, objects_mc, axes_mc, 9000 + i, [0.30])[1][0]
          - recovery.TARGET_ELEVATION for i in range(TRIALS)]
stats = montecarlo.summarise(errors, CONFIDENCE, seed=1)
print(montecarlo.format_summary("elevation error", stats, "degrees"))
print(f"elevation error: {stats['mean']:+.4f} plus or minus {stats['std']:.4f} "
      f"degrees across {TRIALS} trials")


The full 200-trial run, with the camera-baseline sweep and the same treatment applied
to the texture-versus-speckle correlation and to the chirp's range error:


In [ ]:
display(Image(filename=figure("demo13_montecarlo.png"), width=1000))


Panel B is the result worth pausing on. The recovery's bias grows steadily with the
camera's mounting baseline, from essentially zero at 5 cm to about -0.047 degrees at
60 cm. A co-located camera reads elevation straight off the pixel and the sonar's range
error does not enter; the further the camera sits from the head, the more that range
error projects into the recovered angle. Every interval here is tight enough that the
trend is not noise.


## 4. The acoustic chain: chirp, matched filter, range resolution

Everything above forms an image geometrically. This section works in the time domain
instead: a transmitted waveform, echoes delayed by `tau = 2r/c`, noise, and a matched
filter.

A plain pulse of duration `T` cannot separate echoes closer than `c T / 2`. A linear FM
chirp sweeping `B` hertz over the same `T` compresses in the matched filter to a pulse
of width about `1/B`, so the resolution becomes `c / (2B)`, independent of `T`. The
improvement is the time-bandwidth product `B T`.

In [ ]:
chirp_sonar = sonar.ChirpSonar(frequency_hz=300e3, chirp_bandwidth_hz=60e3,
                               chirp_duration_s=2e-3, sample_rate_hz=1.2e6)
print(f"time-bandwidth product BT   {chirp_sonar.time_bandwidth_product:.0f}")
print(f"chirp resolution  c/(2B)    {1000 * chirp_sonar.range_resolution_m:.2f} mm")
print(f"plain pulse       cT/2      {chirp_sonar.uncompressed_resolution_m:.3f} m")

# One reflector, no noise: measure the compressed width against theory.
sweep = chirp_sonar.transmit(swept=True)
solo = acoustics.envelope(
    chirp_sonar.compress(chirp_sonar.receive([(4.0, 1.0, 1.0)], sweep, 0.010), sweep))
ranges = chirp_sonar.range_axis_m(len(solo))
width = acoustics.half_power_width(solo, int(np.argmax(solo))) * (ranges[1] - ranges[0])
print(f"\nmeasured half-power width   {1000 * width:.2f} mm")
print(f"theory 0.886 c/(2B)         {1000 * 0.886 * chirp_sonar.range_resolution_m:.2f} mm")

In [ ]:
display(Image(filename=figure("demo12_chirp.png"), width=1000))

Two targets 50 mm apart come back at -0.62 mm and +0.63 mm, inside one sample. A plain
pulse of the same duration merges them into a single blob.

## 5. Listening to it

The three waveforms below are the real records, resampled onto a time base 600 times
longer. That divides every frequency by 600 and leaves the shape untouched, so the
300 kHz carrier plays at 500 Hz.

In [ ]:
for label, caption in (("transmit", "transmitted chirp, 300 to 360 kHz sweep"),
                       ("received", "received record: two echoes buried in noise"),
                       ("compressed", "after the matched filter: a compressed click")):
    path = figure(f"demo12_chirp_{label}.wav")
    print(caption)
    display(Audio(filename=path))

## Where this stands

Reproduced end to end: the elevation ambiguity to 1.1e-15 of peak, the intensity model
against its closed form to 2.3e-16 relative, shadow-based height recovery to 0.93 mm
rms about a fitted line, opti-acoustic elevation recovery to 0.11 cm of height error
over 200 trials, and pulse compression to 0.2% of the theoretical width.

Not yet built, and deliberately not faked here: delay-and-sum digital beamforming from
the raw element signals, sound-speed refraction, a CFAR detector on the noisy images,
and a transducer response model for the transmitted waveform.
